# 01 — Generator Parameters → Structural Descriptor State

모든 생성구조(non-LHS 포함)를 사용해 **Voxel 생성인자 + continuous latent variables → descriptor latent state**를 학습합니다. 0.10 mm screening과 0.03 mm final은 `meta__fidelity`로 함께 학습합니다.


In [ ]:
from pathlib import Path
import sys,os,json
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd()
if str(CODE_DIR) not in sys.path:sys.path.insert(0,str(CODE_DIR))
RESUME=True;MAX_DESCRIPTOR_COMPONENTS=32;DESCRIPTOR_VARIANCE=0.985;CV_SPLITS=5;RANDOM_SEED=42;N_JOBS=-1

INCLUDE_LEGACY_SEED_ROWS_IN_FORWARD_MODEL=False


In [ ]:
import joblib,pandas as pd
from voxel_ml_common import load_contract,load_all_structures,mark_stage,stage_done
c=load_contract(PROJECT_POINTER);df=load_all_structures(c);out=Path(c['model_root'])/'01_gen2desc';out.mkdir(parents=True,exist_ok=True)
# Permanent copy makes this notebook restartable even if the active Voxel run changes later.
train_csv=out/'training_snapshot.csv'
if not train_csv.exists():df.to_csv(train_csv,index=False,encoding='utf-8-sig')
mark_stage(out,'01_load_data','completed',[train_csv],{'rows':len(df)})
print('Rows:',len(df),'| final:',int((df.get('meta__fidelity','')=='final').sum()),'| screening:',int((df.get('meta__fidelity','')=='screening').sum()))
display(df.head())


In [ ]:
import pandas as pd,joblib
from gen2desc_model import train_gen2desc
from voxel_ml_common import load_contract,mark_stage
c=load_contract(PROJECT_POINTER);out=Path(c['model_root'])/'01_gen2desc';bundle_path=out/'gen2desc_bundle.joblib';df=pd.read_csv(out/'training_snapshot.csv')
if RESUME and bundle_path.exists():
    bundle=joblib.load(bundle_path);metrics=pd.read_csv(out/'cv_metrics.csv');print('RESUME model:',bundle['best_model'])
else:
    bundle,metrics=train_gen2desc(df,out,MAX_DESCRIPTOR_COMPONENTS,DESCRIPTOR_VARIANCE,CV_SPLITS,RANDOM_SEED,N_JOBS,INCLUDE_LEGACY_SEED_ROWS_IN_FORWARD_MODEL)
mark_stage(out,'02_train_model','completed',[bundle_path,out/'cv_metrics.csv',out/'descriptor_reconstruction_audit.csv'],{'best_model':bundle['best_model'],'latent_dim':bundle['descriptor_encoder'].n_components_})
display(metrics);display(pd.read_csv(out/'descriptor_reconstruction_audit.csv').sort_values('r2_from_oof_latent',ascending=False).head(30))


### 산출물
`gen2desc_bundle.joblib`은 이후 모든 Notebook이 공통으로 사용합니다. Descriptor PCA는 단순 LHS용 PCA와 분리된 **영구 structural-state coordinate system**입니다.
